[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# Reading Rows &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup: the classes, `hero_engine`, and `scratch/heroes.db` with the
eight heroes and three teams loaded. Run it first. The tasks only read, so they can be run in any
order, and the last cell removes the scratch folder.


In [1]:
import re
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    version("sqlmodel")
except PackageNotFoundError:                                        # Colab has no SQLModel: install the pinned version
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "sqlmodel==0.0.42"], check=True)

import sqlmodel
from sqlalchemy import event, func, insert, or_
from sqlalchemy.exc import MultipleResultsFound, NoResultFound
from sqlmodel import Field, Session, SQLModel, col, create_engine, select

TEAMS = [                                                           # name, headquarters
    ("Preventers", "Sharp Tower"),
    ("Z-Force", "Sister Margaret's Bar"),
    ("Wakaland Guard", "Grand Palace"),                             # no heroes, for the joins that keep a team anyway
]

HEROES = [                                                          # name, secret name, age, team
    ("Deadpond", "Dive Wilson", None, "Z-Force"),
    ("Spider-Boy", "Pedro Parqueador", 16, "Preventers"),
    ("Rusty-Man", "Tommy Sharp", 48, "Preventers"),
    ("Tarantula", "Natalia Roman-on", 32, "Preventers"),
    ("Black Lion", "Trevor Challa", 35, "Z-Force"),
    ("Dr. Weird", "Steve Weird", 36, "Z-Force"),
    ("Captain North America", "Esteban Rogelios", 93, "Preventers"),
    ("Princess Sure-E", "Sure-E", None, None),                      # on no team
]


def message(error):
    """An error's text, without the memory address or the version link that make no two runs agree."""
    text = re.sub(r"0x[0-9a-f]+", "0x...", str(error))
    return "\n".join(line for line in text.splitlines() if "errors.pydantic.dev" not in line).strip()


def fields(model):
    """A model's values in the order its class declares them, which a loaded object does not keep."""
    return {name: getattr(model, name) for name in type(model).model_fields}


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id")


def hero_engine(path=None, echo=False):
    """The guide's engine: the database in a file, or with no path one in memory, with foreign keys checked."""
    engine = create_engine("sqlite://" if path is None else f"sqlite:///{path}", echo=echo)

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(connection, record):
        cursor = connection.cursor()
        cursor.execute("PRAGMA foreign_keys=ON")
        cursor.close()

    return engine


def build(engine):
    """Create the tables and load the cast, without a session: every notebook starts from the same rows."""
    SQLModel.metadata.create_all(engine)
    with engine.begin() as connection:
        connection.execute(insert(Team), [{"name": name, "headquarters": where} for name, where in TEAMS])
        teams = {name: number for number, (name, _) in enumerate(TEAMS, start=1)}
        connection.execute(insert(Hero), [{"name": name, "secret_name": secret, "age": age,
                                           "team_id": teams.get(team)}
                                          for name, secret, age, team in HEROES])

shutil.rmtree("scratch", ignore_errors=True)                        # a rerun starts from the same eight heroes
Path("scratch").mkdir()
engine = hero_engine("scratch/heroes.db")
build(engine)

with Session(engine) as session:
    print("sqlmodel", sqlmodel.__version__, "|", len(session.exec(select(Hero)).all()), "heroes in",
          len(session.exec(select(Team)).all()), "teams")


sqlmodel 0.0.42 | 8 heroes in 3 teams


**1.** Every hero with an age, oldest first.


In [2]:
with Session(engine) as session:
    oldest_first = select(Hero).where(col(Hero.age).is_not(None)).order_by(col(Hero.age).desc())
    for hero in session.exec(oldest_first):
        print(f"  {hero.name:<22} {hero.age}")


  Captain North America  93
  Rusty-Man              48
  Dr. Weird              36
  Black Lion             35
  Tarantula              32
  Spider-Boy             16


`is_not(None)` is how SQL asks for a value that is there; `!= None` is not the same question and does
not work.


**2.** One hero, and a hero who is not there.


In [3]:
with Session(engine) as session:
    tommy = session.exec(select(Hero).where(Hero.secret_name == "Tommy Sharp")).one()
    print("one()        :", tommy.name)
    print("one_or_none():", session.exec(select(Hero).where(Hero.secret_name == "Nobody At All")).one_or_none())


one()        : Rusty-Man
one_or_none(): None


`one()` is right here because a secret name belongs to exactly one hero; `one_or_none()` answers with
`None` rather than raising, which is what a lookup that may miss wants.


**3.** The Z-Force, or nobody's team, two ways.


In [4]:
with Session(engine) as session:
    with_bars = select(Hero).where((Hero.team_id == 2) | (col(Hero.team_id).is_(None)))
    print("with |  :", sorted(hero.name for hero in session.exec(with_bars)))

    with_or = select(Hero).where(or_(Hero.team_id == 2, col(Hero.team_id).is_(None)))
    print("with or_:", sorted(hero.name for hero in session.exec(with_or)))


with |  : ['Black Lion', 'Deadpond', 'Dr. Weird', 'Princess Sure-E']
with or_: ['Black Lion', 'Deadpond', 'Dr. Weird', 'Princess Sure-E']


The same four heroes either way. `|` needs parentheses around both comparisons; `or_` needs none,
which is why it reads better with more than two.


**4.** How many heroes each team has.


In [5]:
with Session(engine) as session:
    counted = (select(Team.name, func.count(Hero.id))
               .join(Hero, isouter=True)
               .group_by(Team.name)
               .order_by(Team.name))
    for name, heroes in session.exec(counted):
        print(f"  {name:<16} {heroes}")


  Preventers       4
  Wakaland Guard   0
  Z-Force          3


The outer join is what keeps Wakaland Guard, which has nobody: an inner join would have left it out
of the answer entirely, and counting `Hero.id` rather than `*` is what makes its total 0 instead of
1.


**5.** Every hero, beside their headquarters.


In [6]:
with Session(engine) as session:
    placed = select(Hero.name, Team.headquarters).join(Team).order_by(Hero.name)
    for name, headquarters in session.exec(placed):
        print(f"  {name}: {headquarters}")


  Black Lion: Sister Margaret's Bar
  Captain North America: Sharp Tower
  Deadpond: Sister Margaret's Bar
  Dr. Weird: Sister Margaret's Bar
  Rusty-Man: Sharp Tower
  Spider-Boy: Sharp Tower
  Tarantula: Sharp Tower


An inner join leaves out Princess Sure-E, who is on no team, which is what the task asked for. Two
columns come back as rows, unpacked into the two names.


**6.** Heroes whose name contains a fragment.


In [7]:
def heroes_named(session, fragment):
    """The names of the heroes whose name contains a fragment, in alphabetical order."""
    found = select(Hero.name).where(col(Hero.name).contains(fragment)).order_by(Hero.name)
    return session.exec(found).all()


with Session(engine) as session:
    for fragment in ("man", "a", "zzz"):
        print(f"{fragment!r:>6}:", heroes_named(session, fragment))


 'man': ['Rusty-Man']
   'a': ['Black Lion', 'Captain North America', 'Deadpond', 'Rusty-Man', 'Tarantula']
 'zzz': []


`contains` writes a `LIKE` with the fragment between two wildcards. SQLite's `LIKE` ignores case for
plain text, so `man` finds Rusty-Man; on PostgreSQL the same query would be case sensitive, and
`ilike` is what asks for the other behavior there.

Last, the engine lets go of the file, and this cell removes the scratch folder:


In [8]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Reading Rows](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/05-reading-rows.ipynb)  &nbsp;&middot;&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)
